# Method 3 — RAG Dynamic Toxicity Classifier

Few-shot with **per-query** embedding retrieval using `text-embedding-3-small`.
For each comment, the pool is searched at inference time via cosine similarity;
the top-K most similar examples are injected into the prompt.
Hard labels always include at least one extreme example; normal labels use nearest neighbours.
Confidence scores (0-100).

**Prerequisite:** `few_shot_pool.csv` must be present (the training/pool CSV).
On first run the pool embeddings are computed and cached to `pool_embeddings.npz`.


In [6]:
# ============================================================
# IMPORTS
# ============================================================
import os, re, json, time, math, asyncio, hashlib
import numpy as np
import pandas as pd
import tiktoken

from openai import OpenAI, AsyncOpenAI
from sklearn.metrics import (
    precision_recall_fscore_support, accuracy_score,
    hamming_loss, jaccard_score, multilabel_confusion_matrix,
    roc_auc_score
)


In [ ]:
# ============================================================
# CONFIG
# ============================================================

# ── Files ─────────────────────────────────────────────────────
BASE_DIR    = "/Users/dhikarosaripurba/Documents/MSc Business Analytics/06. LLM/1. Group Assignment"
TEST_CSV    = f"{BASE_DIR}/stress_test_eval_set.csv"
POOL_CSV    = f"{BASE_DIR}/few_shot_pool_fixed_3.csv"          # pool to embed & retrieve from
EMBED_CACHE = f"{BASE_DIR}/pool_embeddings.npz"        # cached pool embeddings (auto-created)
OUTPUT_CSV  = f"{BASE_DIR}/predictions_rag_dynamic.csv"

# ── Models ────────────────────────────────────────────────────
GPT_MODEL   = "gpt-4.1"
EMBED_MODEL = "text-embedding-3-small"

# ── Labels ────────────────────────────────────────────────────
LABEL_COLS         = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
HARD_LABELS        = ["severe_toxic", "threat", "identity_hate"]
PRECISION_PRIORITY = ["severe_toxic", "threat", "insult"]

# ── Thresholds (applied to 0-100 confidence scores) ──────────
LABEL_THRESHOLDS = {
    "toxic": 25, "severe_toxic": 75, "obscene": 40,
    "threat": 40, "insult": 50, "identity_hate": 55,
}

# ── RAG retrieval settings ────────────────────────────────────
RAG_K              = 7      # examples retrieved per query
RAG_HARD_K         = 3      # guaranteed examples for hard labels
RAG_SIM_FLOOR      = 0.10   # minimum cosine similarity to include an example
EMBED_BATCH_SIZE   = 512    # rows per embedding API call (pool indexing)

# ── Async settings ────────────────────────────────────────────
ASYNC_CONCURRENCY = 8
CHECKPOINT_EVERY  = 500

# ── Pricing ───────────────────────────────────────────────────
GPT_INPUT_COST_PER_1K  = 0.00080
GPT_OUTPUT_COST_PER_1K = 0.00320
EMBED_COST_PER_1K      = 0.00002

# ── Reproducibility ───────────────────────────────────────────
GPT_SEED = 42

# ── Method info ───────────────────────────────────────────────
METHOD_NAME = "RAG Dynamic (GPT-4.1)"

# ── Clients ───────────────────────────────────────────────────
api_key = os.getenv("OPENAI_API_KEY", "")
if not api_key:
    raise ValueError("OPENAI_API_KEY not set. Run: export OPENAI_API_KEY='sk-...'")

client       = OpenAI(api_key=api_key)
async_client = AsyncOpenAI(api_key=api_key)

try:
    _enc = tiktoken.encoding_for_model(GPT_MODEL)
except KeyError:
    _enc = tiktoken.get_encoding("cl100k_base")


In [8]:
# ============================================================
# UTILITIES
# ============================================================

def seconds_to_hms(s):
    h = int(s // 3600)
    m = int((s % 3600) // 60)
    s = s % 60
    return f"{h:02d}:{m:02d}:{s:05.2f}"

def count_tokens(text):
    if not isinstance(text, str) or not text:
        return 0
    return len(_enc.encode(text))

def clean_text(text, max_chars=12000):
    if text is None:
        return ""
    text = str(text)
    text = text.replace("\x00", " ")
    text = re.sub(r"[\x01-\x08\x0B\x0C\x0E-\x1F\x7F]", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = text.encode("utf-8", "ignore").decode("utf-8", "ignore")
    return text[:max_chars] if len(text) > max_chars else text

def safe_for_api(text):
    if not isinstance(text, str):
        text = str(text)
    text = text.replace("\x00", " ")
    text = re.sub(r"[\x01-\x08\x0B\x0C\x0E-\x1F\x7F-\x9F]", " ", text)
    text = re.sub(r"\\{3,}", " ", text)
    text = re.sub(r"[\ud800-\udfff]", "", text)
    text = re.sub(r"\s{3,}", "  ", text).strip()
    return text

def safe_json_extract(raw):
    m = re.search(r"\{.*\}", raw, flags=re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None

def extract_scores(raw):
    parsed = safe_json_extract(raw)
    if parsed is None:
        return None
    scores = parsed.get("scores", parsed)
    if not isinstance(scores, dict):
        return None
    result = {}
    for c in LABEL_COLS:
        try:
            result[c] = int(float(scores.get(c, 0)))
        except (ValueError, TypeError):
            result[c] = 0
    # If model returned binary 0/1 instead of 0-100, scale up
    if all(v in (0, 1) for v in result.values()):
        result = {c: v * 100 for c, v in result.items()}
    return result

def apply_thresholds(scores, thresholds=None):
    t = thresholds or LABEL_THRESHOLDS
    return {c: int(scores[c] >= t[c]) for c in LABEL_COLS}


In [9]:
# ============================================================
# RETRY WRAPPER
# ============================================================

def call_with_backoff(fn, *args, max_retries=5, base_delay=1.0, **kwargs):
    for attempt in range(max_retries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            err = str(e)
            if "429" in err or "rate_limit" in err.lower() or "timeout" in err.lower():
                delay = base_delay * (2 ** attempt)
                print(f"  [backoff] attempt {attempt+1} — {delay:.1f}s")
                time.sleep(delay)
            elif "400" in err and attempt == 0:
                print(f"  [400-fix] sanitizing and retrying")
                if "messages" in kwargs:
                    kwargs["messages"] = [
                        {k: safe_for_api(v) if isinstance(v, str) else v for k, v in msg.items()}
                        for msg in kwargs["messages"]
                    ]
            else:
                raise
    raise RuntimeError(f"Max retries ({max_retries}) exceeded")


## Build / Load Pool Index (Per-Query Embedding Retrieval)


In [10]:
# ============================================================
# POOL EMBEDDING INDEX
# ============================================================
# Embeds the few-shot pool once (or loads from cache) so that
# at inference time each query comment can retrieve its K
# nearest neighbours via cosine similarity.

def embed_texts(texts, model=EMBED_MODEL, batch_size=EMBED_BATCH_SIZE):
    """Embed a list of texts in batches. Returns (matrix N×D, total_tokens)."""
    all_vecs = []
    total_tokens = 0
    for i in range(0, len(texts), batch_size):
        batch = [safe_for_api(t)[:8000] for t in texts[i:i+batch_size]]
        resp = call_with_backoff(
            client.embeddings.create,
            input=batch,
            model=model,
        )
        all_vecs.extend([d.embedding for d in resp.data])
        total_tokens += resp.usage.total_tokens
        if (i // batch_size) % 5 == 0:
            print(f"  embedded {min(i+batch_size, len(texts)):,}/{len(texts):,} rows ...", end="\r")
    print()
    return np.array(all_vecs, dtype=np.float32), total_tokens


def build_or_load_pool_index(pool_csv, cache_path):
    """
    Load pool CSV, embed all comments, cache to .npz.
    Returns (pool_df, pool_matrix_L2_normalised, embed_token_count).
    """
    pool_df = pd.read_csv(pool_csv)
    pool_df["comment_text"] = pool_df["comment_text"].fillna("").astype(str)
    for c in LABEL_COLS:
        pool_df[c] = pd.to_numeric(pool_df[c], errors="coerce").fillna(0).astype(int)
    if "id" not in pool_df.columns:
        pool_df["id"] = np.arange(len(pool_df)).astype(str)
    print(f"Pool loaded: {len(pool_df):,} rows")

    embed_tokens_used = 0

    if os.path.exists(cache_path):
        print(f"Loading cached pool embeddings from {cache_path} ...")
        cached = np.load(cache_path)
        pool_matrix = cached["matrix"].astype(np.float32)
        print(f"  Loaded matrix: {pool_matrix.shape}")
    else:
        print(f"No cache found — embedding {len(pool_df):,} pool rows with {EMBED_MODEL} ...")
        texts = pool_df["comment_text"].tolist()
        pool_matrix, embed_tokens_used = embed_texts(texts)
        # L2-normalise for fast cosine similarity via dot product
        norms = np.linalg.norm(pool_matrix, axis=1, keepdims=True).clip(min=1e-9)
        pool_matrix = pool_matrix / norms
        np.savez_compressed(cache_path, matrix=pool_matrix)
        print(f"  Saved to {cache_path}  |  embed tokens used: {embed_tokens_used:,}")

    # Ensure matrix is already L2-normalised (norms ≈ 1.0)
    norms = np.linalg.norm(pool_matrix, axis=1)
    if not np.allclose(norms, 1.0, atol=0.01):
        pool_matrix = pool_matrix / norms.clip(min=1e-9)[:, None]

    return pool_df, pool_matrix, embed_tokens_used


def retrieve_examples(query_vec, pool_df, pool_matrix, k=RAG_K,
                       hard_k=RAG_HARD_K, sim_floor=RAG_SIM_FLOOR):
    """
    Given a unit-norm query embedding, return up to k pool rows
    ranked by cosine similarity, with hard-label coverage guarantee.

    Strategy:
      1. Score all pool rows via dot product (= cosine sim, since both normalised).
      2. Filter by sim_floor.
      3. For each HARD_LABEL, guarantee at least one positive example
         from the top-hard_k most similar positives (if any exist above floor).
      4. Fill remaining slots with the highest-sim examples not yet selected.
      5. Always include one clean (is_clean=1) example for contrast if slot available.
    Returns list of dicts: {comment_text, labels, similarity}.
    """
    # Step 1: cosine similarities
    sims = pool_matrix @ query_vec          # shape (N,)

    # Step 2: apply floor
    above_floor = np.where(sims >= sim_floor)[0]
    if len(above_floor) == 0:
        # Fallback: take top-k regardless of floor
        above_floor = np.argsort(sims)[::-1][:k * 3]

    candidates = above_floor[np.argsort(sims[above_floor])[::-1]]  # sorted desc

    selected_idx = []
    seen = set()

    # Step 3: hard-label coverage
    for lbl in HARD_LABELS:
        lbl_mask = pool_df[lbl].values[candidates] == 1
        lbl_candidates = candidates[lbl_mask][:hard_k]
        for idx in lbl_candidates:
            if idx not in seen and len(selected_idx) < k:
                selected_idx.append(idx)
                seen.add(idx)

    # Step 4: fill with top-sim
    for idx in candidates:
        if len(selected_idx) >= k:
            break
        if idx not in seen:
            selected_idx.append(idx)
            seen.add(idx)

    # Step 5: contrast clean example (if slot available)
    if "is_clean" in pool_df.columns:
        clean_mask = pool_df["is_clean"].values[candidates] == 1
        clean_candidates = candidates[clean_mask]
        for idx in clean_candidates:
            if idx not in seen:
                selected_idx.append(idx)
                seen.add(idx)
                break  # one clean example is enough

    # Build output list
    examples = []
    for idx in selected_idx:
        row = pool_df.iloc[int(idx)]
        examples.append({
            "comment_text": str(row["comment_text"])[:600],
            "labels": {c: int(row[c]) for c in LABEL_COLS},
            "similarity": float(sims[idx]),
        })
    return examples


# ── Build index (runs once; uses cache on subsequent runs) ─────────────────
print("Building / loading pool index...")
POOL_DF, POOL_MATRIX, _pool_embed_tokens = build_or_load_pool_index(POOL_CSV, EMBED_CACHE)
print(f"Index ready: {POOL_MATRIX.shape[0]:,} vectors × {POOL_MATRIX.shape[1]}d")
if _pool_embed_tokens:
    _pool_embed_cost = (_pool_embed_tokens / 1000) * EMBED_COST_PER_1K
    print(f"Pool embedding cost: ${_pool_embed_cost:.4f}  ({_pool_embed_tokens:,} tokens)")


Building / loading pool index...
Pool loaded: 151,543 rows
No cache found — embedding 151,543 pool rows with text-embedding-3-small ...
  embedded 151,543/151,543 rows ...
  Saved to /Users/dhikarosaripurba/Documents/MSc Business Analytics/06. LLM/1. Group Assignment/pool_embeddings.npz  |  embed tokens used: 13,795,220
Index ready: 151,543 vectors × 1536d
Pool embedding cost: $0.2759  (13,795,220 tokens)


## Dynamic RAG Prompt Builder


In [11]:
def embed_single(text, model=EMBED_MODEL):
    """
    Embed a single comment and return (unit-norm vector, token_count).
    Called at inference time for every query row.
    """
    safe_text = safe_for_api(text)[:8000]
    resp = call_with_backoff(
        client.embeddings.create,
        input=[safe_text],
        model=model,
    )
    vec = np.array(resp.data[0].embedding, dtype=np.float32)
    norm = np.linalg.norm(vec)
    vec = vec / norm.clip(min=1e-9)   # L2-normalise
    tokens = resp.usage.total_tokens
    return vec, tokens


def format_dynamic_examples(examples):
    """Format retrieved examples into the prompt block."""
    blocks = []
    for ex in examples:
        scores = {k: v * 100 for k, v in ex["labels"].items()}
        sim_pct = round(ex["similarity"] * 100, 1)
        blocks.append(
            f"Comment (sim={sim_pct}%): {ex['comment_text']}\n"
            f"Scores: {json.dumps(scores)}"
        )
    return "\n\n".join(blocks)


def build_rag_dynamic_prompt(row):
    """
    Build a RAG prompt with per-query retrieved examples.

    Steps:
      1. Embed the query comment via text-embedding-3-small.
      2. Retrieve top-K most similar pool examples (cosine sim).
      3. Inject retrieved examples into the prompt.

    Returns (prompt, system_msg, embed_tokens).
    """
    comment = clean_text(str(row.comment_text))

    # ── Step 1: embed query ─────────────────────────────────────
    query_vec, embed_tokens = embed_single(comment)

    # ── Step 2: retrieve nearest neighbours ─────────────────────
    examples = retrieve_examples(
        query_vec, POOL_DF, POOL_MATRIX,
        k=RAG_K, hard_k=RAG_HARD_K, sim_floor=RAG_SIM_FLOOR,
    )

    # ── Step 3: format retrieved block ──────────────────────────
    retrieved_block = format_dynamic_examples(examples)
    n_retrieved = len(examples)

    system_msg = (
        "You are a precise multi-label toxicity classification assistant. "
        "Return valid JSON only."
    )

    json_template = """{\n  \"scores\": {\n    \"toxic\": 0,\n    \"severe_toxic\": 0,\n    \"obscene\": 0,\n    \"threat\": 0,\n    \"insult\": 0,\n    \"identity_hate\": 0\n  }\n}"""

    prompt = f"""You are an expert multi-label toxicity classifier for online comments.

Your task is to assign a confidence score from 0 to 100 for each toxicity label.

LABELS:
- toxic: rude, aggressive, hostile, or generally offensive language
- severe_toxic: extreme abuse, dehumanising hostility, or very intense toxic language
- obscene: explicit profanity, vulgarity, or sexually explicit/offensive wording
- threat: statement implying physical harm, violence, intimidation, or a clear wish to injure
- insult: direct personal attack, humiliation, ridicule, or degrading statement toward a person
- identity_hate: attack, contempt, or hate directed at a protected group identity
  (e.g. race, religion, ethnicity, nationality, gender, sexual orientation)

IMPORTANT DECISION RULES:
1. Multi-label classification is allowed. More than one label may be present.
2. severe_toxic is a strict subset of toxic.
   - If severe_toxic is high, toxic should also be high.
3. threat requires a genuine implication of harm or violence.
   - Mere anger, profanity, or \"I hate you\" is not enough.
4. identity_hate requires hostility toward a protected group identity.
   - Attacking one individual without group-based targeting is not identity_hate.
5. Profanity alone may indicate obscene and/or toxic, but not automatically severe_toxic.
6. insult requires a direct degrading or attacking expression toward a person or target.
7. Be conservative for rare labels, especially threat and identity_hate.
8. Base the decision only on the text provided. Do not infer hidden intent beyond the wording.

HOW TO USE THE REFERENCE EXAMPLES:
- The {n_retrieved} examples below were retrieved specifically for this comment
  using semantic similarity (text-embedding-3-small cosine search).
- Similarity scores are shown — higher means more semantically related.
- Use them to calibrate label thresholds, not to copy scores mechanically.

RETRIEVED EXAMPLES (ordered by similarity):
{retrieved_block}

Now classify this comment:
\"\"\"{comment}\"\"\"

SCORING GUIDANCE:
- 0-10: label clearly absent
- 11-39: weak or ambiguous evidence
- 40-59: borderline / uncertain presence
- 60-79: reasonably clear presence
- 80-100: very clear presence

OUTPUT REQUIREMENTS:
- Return ONLY one valid JSON object.
- Use EXACTLY the keys shown below.
- Do not include explanations, markdown, or extra text.

{json_template}
"""

    return prompt, system_msg, embed_tokens


In [12]:
# ============================================================
# INFERENCE ENGINE
# ============================================================

def gpt_classify(prompt, system_msg="You are a precise toxicity classification assistant."):
    """Call GPT and return raw response + token counts."""
    safe_prompt = safe_for_api(prompt)
    safe_sys    = safe_for_api(system_msg)
    resp = call_with_backoff(
        client.chat.completions.create,
        model=GPT_MODEL,
        temperature=0,
        seed=GPT_SEED,
        max_tokens=150,
        messages=[
            {"role": "system", "content": safe_sys},
            {"role": "user",   "content": safe_prompt},
        ]
    )
    raw     = resp.choices[0].message.content.strip()
    in_tok  = count_tokens(safe_prompt) + count_tokens(safe_sys)
    out_tok = count_tokens(raw)
    return raw, in_tok, out_tok


async def process_row(row, build_prompt_fn, semaphore):
    """
    Process a single row:
      1. Call build_prompt_fn → embeds query + retrieves examples (real embed tokens).
      2. Call GPT with the dynamic prompt.
      3. Extract scores, apply thresholds.
    embed_tokens is now a real count from the embedding API call.
    """
    loop = asyncio.get_event_loop()
    row_start = time.perf_counter()

    # ── Step 1: embed + retrieve (runs in thread to avoid blocking event loop) ──
    async with semaphore:
        prompt, system_msg, embed_tokens = await loop.run_in_executor(
            None, build_prompt_fn, row
        )

        # ── Step 2: GPT classification ──────────────────────────────────────
        raw, in_tok, out_tok = await loop.run_in_executor(
            None, gpt_classify, prompt, system_msg
        )

    # ── Step 3: parse + fallback ─────────────────────────────────────────
    scores = extract_scores(raw)
    if scores is None:
        fallback = (
            f"Classify this comment for toxicity. Return ONLY valid JSON with integer scores 0-100:\n"
            f"{clean_text(str(row.comment_text))}\n\n"
            f'{{"scores":{{"toxic":0,"severe_toxic":0,"obscene":0,"threat":0,"insult":0,"identity_hate":0}}}}'
        )
        async with semaphore:
            raw2, in2, out2 = await loop.run_in_executor(
                None, gpt_classify, fallback, "You are a JSON-only classifier."
            )
        in_tok += in2; out_tok += out2
        scores = extract_scores(raw2) or {c: 0 for c in LABEL_COLS}
        raw = raw2

    pred = apply_thresholds(scores)
    row_elapsed = time.perf_counter() - row_start

    out = {
        "id":                str(getattr(row, "id", row.Index)),
        "comment_text":      row.comment_text,
        "raw_response":      raw,
        "scores":            json.dumps(scores, ensure_ascii=False),
        "embed_tokens":      embed_tokens,   # ← real per-query embedding token count
        "gpt_input_tokens":  in_tok,
        "gpt_output_tokens": out_tok,
        "row_runtime_sec":   row_elapsed,
    }
    for c in LABEL_COLS:
        out[f"true_{c}"] = int(getattr(row, c, 0))
        out[f"pred_{c}"] = int(pred[c])

    return out


async def run_inference(df, build_prompt_fn):
    """Run async inference over all rows with checkpoint/resume."""
    t0 = time.perf_counter()

    print("=" * 72)
    print(f"INFERENCE — {METHOD_NAME}")
    print(f"  Model      : {GPT_MODEL}")
    print(f"  Embed model: {EMBED_MODEL}")
    print(f"  RAG K      : {RAG_K} examples per query")
    print(f"  Rows       : {len(df):,}")
    print(f"  Concurrency: {ASYNC_CONCURRENCY}")
    print("=" * 72)

    # Resume support
    if os.path.exists(OUTPUT_CSV):
        done_df  = pd.read_csv(OUTPUT_CSV)
        done_ids = set(done_df["id"].astype(str))
        rows     = done_df.to_dict(orient="records")
        print(f"Resuming from {len(rows):,} completed rows")
    else:
        done_ids = set()
        rows     = []

    pending = [r for r in df.itertuples() if str(getattr(r, "id", r.Index)) not in done_ids]
    print(f"Rows to process: {len(pending):,}")

    semaphore = asyncio.Semaphore(ASYNC_CONCURRENCY)
    tasks = [process_row(row, build_prompt_fn, semaphore) for row in pending]

    completed = 0
    for coro in asyncio.as_completed(tasks):
        result = await coro
        rows.append(result)
        completed += 1
        if completed % CHECKPOINT_EVERY == 0:
            pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False)
            embed_cost = sum(r.get("embed_tokens", 0) for r in rows) / 1000 * EMBED_COST_PER_1K
            gpt_cost   = (
                sum(r.get("gpt_input_tokens", 0) for r in rows) / 1000 * GPT_INPUT_COST_PER_1K +
                sum(r.get("gpt_output_tokens", 0) for r in rows) / 1000 * GPT_OUTPUT_COST_PER_1K
            )
            print(f"  [{len(rows):>6,}/{len(df):<6,}] embed~${embed_cost:.4f}  gpt~${gpt_cost:.4f}  total~${embed_cost+gpt_cost:.4f}")

    pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False)

    elapsed = time.perf_counter() - t0
    print(f"\nInference complete. {len(rows):,} rows | {seconds_to_hms(elapsed)}")
    print(f"Saved to: {OUTPUT_CSV}")
    print("=" * 72)


In [13]:
# ============================================================
# UNIFIED EVALUATION (matches v4 output format)
# ============================================================

def evaluate_predictions(csv_path=None, method_name=None):
    """
    Unified evaluation matching v4 format.
    Works for all methods — zero-shot, few-shot, RAG static, RAG label-aware.
    """
    csv_file = csv_path or OUTPUT_CSV
    _method  = method_name or METHOD_NAME
    eval_start = time.perf_counter()

    if not os.path.exists(csv_file):
        raise FileNotFoundError(f"{csv_file} not found.")

    df = pd.read_csv(csv_file)
    y_true = df[[f"true_{c}" for c in LABEL_COLS]].values.astype(int)
    y_pred = df[[f"pred_{c}" for c in LABEL_COLS]].values.astype(int)

    # ── Global metrics ────────────────────────────────────────
    micro_p, micro_r, micro_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="micro", zero_division=0)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)

    exact_acc  = accuracy_score(y_true, y_pred)
    h_loss     = hamming_loss(y_true, y_pred)
    jacc_macro = jaccard_score(y_true, y_pred, average="macro", zero_division=0)
    jacc_samp  = jaccard_score(y_true, y_pred, average="samples", zero_division=0)

    # ── Per-label metrics ─────────────────────────────────────
    per_p, per_r, per_f1, per_sup = precision_recall_fscore_support(y_true, y_pred, average=None, zero_division=0)
    mcm = multilabel_confusion_matrix(y_true, y_pred)

    label_acc = {}
    for i, label in enumerate(LABEL_COLS):
        label_acc[label] = float(np.mean(y_true[:, i] == y_pred[:, i]))

    # ── ROC-AUC ───────────────────────────────────────────────
    has_scores = "scores" in df.columns
    per_auc = {}
    micro_auc = None
    macro_auc = None

    if has_scores:
        score_matrix = np.zeros((len(df), len(LABEL_COLS)), dtype=float)
        for j, label in enumerate(LABEL_COLS):
            score_matrix[:, j] = df["scores"].apply(
                lambda s, l=label: json.loads(s).get(l, 0) if isinstance(s, str) else 0
            ).values.astype(float)

        for j, label in enumerate(LABEL_COLS):
            if y_true[:, j].sum() > 0 and y_true[:, j].sum() < len(y_true):
                per_auc[label] = roc_auc_score(y_true[:, j], score_matrix[:, j])
            else:
                per_auc[label] = float("nan")

        valid_labels = [j for j, l in enumerate(LABEL_COLS) if not np.isnan(per_auc.get(l, float("nan")))]
        if valid_labels:
            try:
                micro_auc = roc_auc_score(y_true[:, valid_labels], score_matrix[:, valid_labels], average="micro")
            except ValueError:
                micro_auc = None
            macro_auc = float(np.nanmean([per_auc[LABEL_COLS[j]] for j in valid_labels]))

    # ── Operational metrics ───────────────────────────────────
    gpt_in_tok  = int(df["gpt_input_tokens"].fillna(0).sum())  if "gpt_input_tokens"  in df.columns else 0
    gpt_out_tok = int(df["gpt_output_tokens"].fillna(0).sum()) if "gpt_output_tokens" in df.columns else 0
    embed_tok   = int(df["embed_tokens"].fillna(0).sum())       if "embed_tokens"      in df.columns else 0
    mean_rt     = float(df["row_runtime_sec"].fillna(0).mean()) if "row_runtime_sec"   in df.columns else 0.0
    total_rt    = float(df["row_runtime_sec"].fillna(0).sum())  if "row_runtime_sec"   in df.columns else 0.0

    embed_cost = (embed_tok / 1000)   * EMBED_COST_PER_1K
    gin_cost   = (gpt_in_tok / 1000)  * GPT_INPUT_COST_PER_1K
    gout_cost  = (gpt_out_tok / 1000) * GPT_OUTPUT_COST_PER_1K
    total_cost = embed_cost + gin_cost + gout_cost

    cost_per_sample = total_cost / len(df) if len(df) > 0 else 0.0
    throughput      = len(df) / total_rt if total_rt > 0 else 0.0

    # ── Print report ──────────────────────────────────────────
    W = 78
    print(f"\n{'=' * W}")
    print(f"   {_method.upper()} — EVAL REPORT")
    print(f"{'=' * W}")
    print(f"  Generated   : {time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"  GPT model   : {GPT_MODEL}")
    print(f"  Method      : {_method}")
    print(f"  Rows eval   : {len(df):,}")
    print()

    print("── GLOBAL METRICS ──")
    print(f"  Micro     Prec={micro_p:.4f}  Rec={micro_r:.4f}  F1={micro_f1:.4f}")
    print(f"  Macro     Prec={macro_p:.4f}  Rec={macro_r:.4f}  F1={macro_f1:.4f}")
    print(f"  Weighted  Prec={weighted_p:.4f}  Rec={weighted_r:.4f}  F1={weighted_f1:.4f}")
    print(f"  Exact Match Acc  : {exact_acc:.4f}")
    print(f"  Hamming Loss     : {h_loss:.4f}")
    print(f"  Jaccard(macro)   : {jacc_macro:.4f}")
    print(f"  Jaccard(samples) : {jacc_samp:.4f}")
    if micro_auc is not None: print(f"  ROC-AUC(micro)   : {micro_auc:.4f}")
    if macro_auc is not None: print(f"  ROC-AUC(macro)   : {macro_auc:.4f}")
    print()

    print("── PER-LABEL METRICS ──\n")
    hdr = f"  {'Label':<18}{'Prec':>8}{'Rec':>8}{'F1':>8}{'AUC':>8}{'Acc':>8}{'TP':>6}{'FP':>6}{'FN':>6}{'TN':>6}{'Supp':>6}"
    print(hdr)
    print("  " + "─" * (len(hdr) - 2))
    for i, label in enumerate(LABEL_COLS):
        tn, fp, fn, tp = mcm[i].ravel()
        star = " ★" if label in PRECISION_PRIORITY else ""
        auc_str = f"{per_auc[label]:.4f}" if label in per_auc and not np.isnan(per_auc.get(label, float('nan'))) else "  n/a "
        print(f"  {label + star:<18}{per_p[i]:>8.4f}{per_r[i]:>8.4f}{per_f1[i]:>8.4f}{auc_str:>8}{label_acc[label]:>8.4f}{tp:>6}{fp:>6}{fn:>6}{tn:>6}{int(per_sup[i]):>6}")
    print(f"\n  Note: Label-wise Acc can be misleading on imbalanced labels — F1 and AUC are more reliable.")
    print()

    print("── OPERATIONAL METRICS ──")
    print(f"  Mean latency      : {mean_rt:.4f}s per row")
    print(f"  Throughput        : {throughput:.2f} rows/sec")
    print()

    print("── COST & TIME ──")
    if embed_tok > 0:
        print(f"  Embed tokens    : {embed_tok:,}   cost ~${embed_cost:.6f}")
    print(f"  GPT input tok.  : {gpt_in_tok:,}   cost ~${gin_cost:.6f}")
    print(f"  GPT output tok. : {gpt_out_tok:,}   cost ~${gout_cost:.6f}")
    print(f"  Total cost      : ${total_cost:.6f}")
    print(f"  Cost per sample : ${cost_per_sample:.6f}")
    print(f"  Eval compute    : {seconds_to_hms(time.perf_counter() - eval_start)}")
    print()
    print("✅ Evaluation complete.")
    print("=" * W)

    return {
        "method": _method, "n_rows": len(df),
        "micro_f1": micro_f1, "macro_f1": macro_f1, "weighted_f1": weighted_f1,
        "exact_match_acc": exact_acc, "hamming_loss": h_loss,
        "jaccard_samples": jacc_samp, "roc_auc_macro": macro_auc,
        "cost_per_sample": cost_per_sample, "throughput": throughput, "mean_latency": mean_rt,
    }


## Run Pipeline

In [15]:
# Load data
df = pd.read_csv(TEST_CSV).dropna(subset=["comment_text"]).copy()
df["comment_text"] = df["comment_text"].astype(str)
for c in LABEL_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)
if "id" not in df.columns:
    df["id"] = np.arange(len(df)).astype(str)
else:
    df["id"] = df["id"].astype(str)

print(f"Loaded {len(df):,} rows from {TEST_CSV}")


Loaded 8,000 rows from /Users/dhikarosaripurba/Documents/MSc Business Analytics/06. LLM/1. Group Assignment/stress_test_eval_set.csv


In [16]:
# ── Run inference + evaluation ─────────────────────────────────
await run_inference(df, build_rag_dynamic_prompt)

# ── Evaluate (ROC-AUC included automatically) ─────────────────
print()
result = evaluate_predictions()


INFERENCE — RAG Dynamic (GPT-4.1)
  Model      : gpt-4.1
  Embed model: text-embedding-3-small
  RAG K      : 7 examples per query
  Rows       : 8,000
  Concurrency: 8
Rows to process: 8,000
  [   500/8,000 ] embed~$0.0008  gpt~$0.7213  total~$0.7221
  [ 1,000/8,000 ] embed~$0.0017  gpt~$1.4499  total~$1.4516
  [ 1,500/8,000 ] embed~$0.0027  gpt~$2.1808  total~$2.1836
  [ 2,000/8,000 ] embed~$0.0036  gpt~$2.9028  total~$2.9064
  [ 2,500/8,000 ] embed~$0.0044  gpt~$3.6214  total~$3.6258
  [ 3,000/8,000 ] embed~$0.0054  gpt~$4.3472  total~$4.3526
  [ 3,500/8,000 ] embed~$0.0063  gpt~$5.0717  total~$5.0780
  [ 4,000/8,000 ] embed~$0.0073  gpt~$5.7983  total~$5.8056
  [ 4,500/8,000 ] embed~$0.0081  gpt~$6.5202  total~$6.5283
  [400-fix] sanitizing and retrying
  [ 5,000/8,000 ] embed~$0.0090  gpt~$7.2418  total~$7.2507
  [ 5,500/8,000 ] embed~$0.0102  gpt~$7.9766  total~$7.9868
  [ 6,000/8,000 ] embed~$0.0110  gpt~$8.6965  total~$8.7075
  [ 6,500/8,000 ] embed~$0.0118  gpt~$9.4115  total~